# Data Quality and Quarantine

Applies business validation rules to the six cleaned Silver staging tables from Notebook 04, quarantines invalid records with rejection reasons, and saves valid records as the final Silver Delta tables.

* **ERROR** rules reject a record into its quarantine table.
* **WARNING** rules keep the record but flag it in `_dq_warnings` (used for systematic issues that need a business decision, e.g. unmatched warehouse IDs).

In [0]:
# Step 1 - Read the six cleaned staging tables from Notebook 04
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER = "logistics.silver"

orders     = spark.table(f"{SILVER}.clean_shipment_orders")
tracking   = spark.table(f"{SILVER}.clean_shipment_tracking_events")
attempts   = spark.table(f"{SILVER}.clean_delivery_attempts")
carriers   = spark.table(f"{SILVER}.clean_carrier_master")
warehouses = spark.table(f"{SILVER}.clean_warehouse_master")
routes     = spark.table(f"{SILVER}.clean_route_master")

# Business rule configuration
VALID_SHIPMENT_STATUS = ["CREATED", "IN_TRANSIT", "OUT_FOR_DELIVERY", "DELIVERED"]
VALID_SERVICE_LEVELS  = ["STANDARD", "EXPRESS", "SAME_DAY"]
VALID_EVENT_TYPES     = ["PICKED_UP", "AT_HUB", "OUT_FOR_DELIVERY", "DELIVERED"]
VALID_ATTEMPT_RESULTS = ["DELIVERED", "FAILED", "RESCHEDULED"]
VALID_FLAGS           = ["Y", "N"]
MAX_WEIGHT_KG   = 1000    # 999999.99 is a sentinel value in the source
MAX_DISTANCE_KM = 5000
MAX_PLANNED_HRS = 72

# Reference keys for referential integrity
carrier_ids   = carriers.select("carrier_id").distinct().withColumn("_carrier_found", F.lit(True))
warehouse_ids = warehouses.select(F.col("warehouse_id").alias("origin_warehouse_id")).distinct().withColumn("_warehouse_found", F.lit(True))
route_keys    = routes.select("origin_warehouse_id", "destination_city").distinct().withColumn("_route_found", F.lit(True))
shipment_ref  = (orders.groupBy("shipment_id")
                       .agg(F.min("created_ts").alias("_shipment_created_ts"))
                       .withColumn("_shipment_found", F.lit(True)))

display(spark.createDataFrame(
    [(n, d.count()) for n, d in [("shipment_orders", orders), ("shipment_tracking_events", tracking),
                                 ("delivery_attempts", attempts), ("carrier_master", carriers),
                                 ("warehouse_master", warehouses), ("route_master", routes)]],
    "dataset string, input_rows long"))

In [0]:
# Rules engine: every rule returns a struct when the record FAILS it, else NULL

def rule(rule_id, category, severity, failing):
    return F.when(failing, F.struct(F.lit(rule_id).alias("rule_id"),
                                    F.lit(category).alias("category"),
                                    F.lit(severity).alias("severity")))

def not_null(cols):                      # Step 3 - NULL validation
    return [rule(f"NULL_{c.upper()}", "NULL", "ERROR", F.col(c).isNull()) for c in cols]

def id_format(patterns):                 # Step 2 - mandatory-field validation (format of required IDs)
    return [rule(f"BAD_FORMAT_{c.upper()}", "MANDATORY", "ERROR",
                 F.col(c).isNotNull() & ~F.col(c).rlike(p)) for c, p in patterns.items()]

def cast_failed():                       # Step 2 - failed type conversion from Notebook 04
    return [rule("CAST_FAILURE", "MANDATORY", "ERROR", F.size("_cast_errors") > 0)]

def not_future(cols):                    # Step 5 - timestamp validation
    return [rule(f"FUTURE_{c.upper()}", "TIMESTAMP", "ERROR", F.col(c) > F.current_timestamp()) for c in cols]

def duplicate(order_cols):               # Step 6 - duplicate validation (keep first, reject the rest)
    return [rule("DUPLICATE_KEY", "DUPLICATE", "ERROR", F.col("_dup_rank") > 1)]

def with_dup_rank(df, keys, order_cols):
    w = Window.partitionBy(*keys).orderBy(*order_cols)
    return df.withColumn("_dup_rank", F.row_number().over(w))

def apply_rules(df, rules):
    return (df.withColumn("_dq_issues", F.array_compact(F.array(*rules)))
              .withColumn("_is_valid", ~F.exists("_dq_issues", lambda x: x["severity"] == "ERROR")))

validated = {}   # dataset -> (validated dataframe, record_id column)

In [0]:
# Shipment orders - Steps 2-7
so = (with_dup_rank(orders, ["shipment_id"], ["_ingestion_timestamp", "_source_file"])
        .join(F.broadcast(carrier_ids), "carrier_id", "left")
        .join(F.broadcast(warehouse_ids), "origin_warehouse_id", "left")
        .join(F.broadcast(route_keys), ["origin_warehouse_id", "destination_city"], "left"))

so_rules = (
    id_format({"shipment_id": r"^SHIPMEN\d+$", "order_id": r"^ORDER\d+$", "customer_id": r"^CUSTOME\d+$",
               "carrier_id": r"^CARRIE\d+$", "origin_warehouse_id": r"^ORIGIN_\d+$"})
  + cast_failed()
  + not_null(["shipment_id", "order_id", "customer_id", "carrier_id", "origin_warehouse_id",
              "created_ts", "promised_delivery_ts", "package_weight_kg", "shipping_cost", "shipment_status"])
  + [rule("INVALID_SHIPMENT_STATUS", "DOMAIN", "ERROR",
          F.col("shipment_status").isNotNull() & ~F.col("shipment_status").isin(VALID_SHIPMENT_STATUS)),
     rule("MISSING_DESTINATION_CITY", "NULL", "WARNING", F.col("destination_city").isNull()),
     rule("INVALID_SERVICE_LEVEL", "DOMAIN", "WARNING",
          F.col("service_level").isNull() | ~F.col("service_level").isin(VALID_SERVICE_LEVELS))]
  # Step 4 - range validation
  + [rule("NON_POSITIVE_WEIGHT", "RANGE", "ERROR", F.col("package_weight_kg") <= 0),
     rule("EXTREME_WEIGHT", "RANGE", "ERROR", F.col("package_weight_kg") > MAX_WEIGHT_KG),
     rule("NEGATIVE_COST", "RANGE", "ERROR", F.col("shipping_cost") < 0)]
  # Step 5 - timestamp validation
  + not_future(["created_ts"])
  + [rule("PROMISED_BEFORE_CREATED", "TIMESTAMP", "ERROR", F.col("promised_delivery_ts") < F.col("created_ts"))]
  # Step 6 - duplicates
  + duplicate(["_ingestion_timestamp"])
  # Step 7 - referential integrity
  + [rule("UNKNOWN_CARRIER", "REFERENTIAL", "ERROR", F.col("carrier_id").isNotNull() & F.col("_carrier_found").isNull()),
     rule("UNKNOWN_WAREHOUSE", "REFERENTIAL", "WARNING", F.col("_warehouse_found").isNull()),
     rule("NO_MATCHING_ROUTE", "REFERENTIAL", "WARNING", F.col("_route_found").isNull())]
)

validated["shipment_orders"] = (apply_rules(so, so_rules)
                                .drop("_carrier_found", "_warehouse_found", "_route_found", "_dup_rank"), "shipment_id")
display(validated["shipment_orders"][0].groupBy("_is_valid").count())

In [0]:
# Shipment tracking events - Steps 2-7
te = (with_dup_rank(tracking, ["tracking_event_id"], ["_ingestion_timestamp", "received_ts"])
        .join(F.broadcast(carrier_ids), "carrier_id", "left")
        .join(shipment_ref, "shipment_id", "left"))

te_rules = (
    id_format({"tracking_event_id": r"^TRACKIN\d+$", "shipment_id": r"^SHIPMEN\d+$",
               "facility_id": r"^FACILIT\d+$", "carrier_id": r"^CARRIE\d+$"})
  + cast_failed()
  + not_null(["tracking_event_id", "shipment_id", "event_ts", "event_type", "carrier_id", "received_ts"])
  + [rule("INVALID_EVENT_TYPE", "DOMAIN", "ERROR",
          F.col("event_type").isNotNull() & ~F.col("event_type").isin(VALID_EVENT_TYPES))]
  # Step 5 - timestamp validation
  + not_future(["event_ts", "received_ts"])
  + [rule("RECEIVED_BEFORE_EVENT", "TIMESTAMP", "ERROR", F.col("received_ts") < F.col("event_ts")),
     rule("EVENT_BEFORE_SHIPMENT_CREATED", "TIMESTAMP", "WARNING", F.col("event_ts") < F.col("_shipment_created_ts"))]
  # Step 6 - duplicates
  + duplicate(["_ingestion_timestamp"])
  # Step 7 - referential integrity
  + [rule("UNKNOWN_SHIPMENT", "REFERENTIAL", "ERROR", F.col("shipment_id").isNotNull() & F.col("_shipment_found").isNull()),
     rule("UNKNOWN_CARRIER", "REFERENTIAL", "ERROR", F.col("carrier_id").isNotNull() & F.col("_carrier_found").isNull())]
)

validated["shipment_tracking_events"] = (apply_rules(te, te_rules)
    .drop("_carrier_found", "_shipment_found", "_shipment_created_ts", "_dup_rank"), "tracking_event_id")
display(validated["shipment_tracking_events"][0].groupBy("_is_valid").count())

In [0]:
# Delivery attempts - Steps 2-7
w_seq = Window.partitionBy("shipment_id").orderBy("attempt_ts", "attempt_id")
da = (with_dup_rank(attempts, ["attempt_id"], ["_ingestion_timestamp", "received_ts"])
        .withColumn("_expected_seq", F.row_number().over(w_seq))
        .join(shipment_ref, "shipment_id", "left"))

da_rules = (
    id_format({"attempt_id": r"^ATTEMPT\d+$", "shipment_id": r"^SHIPMEN\d+$", "driver_id": r"^DRIVER\d+$"})
  + cast_failed()
  + not_null(["attempt_id", "shipment_id", "attempt_ts", "attempt_result", "attempt_sequence", "received_ts"])
  + [rule("INVALID_ATTEMPT_RESULT", "DOMAIN", "ERROR",
          F.col("attempt_result").isNotNull() & ~F.col("attempt_result").isin(VALID_ATTEMPT_RESULTS)),
     rule("FAILED_WITHOUT_REASON", "DOMAIN", "ERROR",
          (F.col("attempt_result") == "FAILED") & F.col("failure_reason").isNull()),
     rule("DELIVERED_WITH_FAILURE_REASON", "DOMAIN", "WARNING",
          (F.col("attempt_result") == "DELIVERED") & F.col("failure_reason").isNotNull()),
     rule("DELIVERED_WITHOUT_POD", "DOMAIN", "WARNING",
          (F.col("attempt_result") == "DELIVERED") & (F.col("proof_of_delivery") != "Y"))]
  # Step 4 - range validation
  + [rule("NON_POSITIVE_SEQUENCE", "RANGE", "ERROR", F.col("attempt_sequence") < 1),
     rule("SEQUENCE_GAP", "RANGE", "WARNING", F.col("attempt_sequence") != F.col("_expected_seq"))]
  # Step 5 - timestamp validation
  + not_future(["attempt_ts", "received_ts"])
  + [rule("RECEIVED_BEFORE_ATTEMPT", "TIMESTAMP", "ERROR", F.col("received_ts") < F.col("attempt_ts")),
     rule("ATTEMPT_BEFORE_SHIPMENT_CREATED", "TIMESTAMP", "WARNING", F.col("attempt_ts") < F.col("_shipment_created_ts"))]
  # Step 6 - duplicates
  + duplicate(["_ingestion_timestamp"])
  # Step 7 - referential integrity
  + [rule("UNKNOWN_SHIPMENT", "REFERENTIAL", "ERROR", F.col("shipment_id").isNotNull() & F.col("_shipment_found").isNull())]
)

validated["delivery_attempts"] = (apply_rules(da, da_rules)
    .drop("_shipment_found", "_shipment_created_ts", "_expected_seq", "_dup_rank"), "attempt_id")
display(validated["delivery_attempts"][0].groupBy("_is_valid").count())

In [0]:
# Master data - Steps 2-7
flag_rule = lambda: [rule("INVALID_ACTIVE_FLAG", "DOMAIN", "ERROR",
                          F.col("active_flag").isNull() | ~F.col("active_flag").isin(VALID_FLAGS))]

# Carrier master
cm = with_dup_rank(carriers, ["carrier_id"], ["_ingestion_timestamp"])
cm_rules = (id_format({"carrier_id": r"^CARRIE\d+$"}) + cast_failed()
          + not_null(["carrier_id", "carrier_name", "onboarded_date"]) + flag_rule()
          + [rule("FUTURE_ONBOARDED_DATE", "TIMESTAMP", "ERROR", F.col("onboarded_date") > F.current_date())]
          + duplicate(["_ingestion_timestamp"]))
validated["carrier_master"] = (apply_rules(cm, cm_rules).drop("_dup_rank"), "carrier_id")

# Warehouse master
wm = with_dup_rank(warehouses, ["warehouse_id"], ["_ingestion_timestamp"])
wm_rules = (id_format({"warehouse_id": r"^WAREHO\d+$"}) + cast_failed()
          + not_null(["warehouse_id", "warehouse_name", "city", "capacity_units", "opening_date"]) + flag_rule()
          + [rule("NON_POSITIVE_CAPACITY", "RANGE", "ERROR", F.col("capacity_units") <= 0),
             rule("FUTURE_OPENING_DATE", "TIMESTAMP", "ERROR", F.col("opening_date") > F.current_date())]
          + duplicate(["_ingestion_timestamp"]))
validated["warehouse_master"] = (apply_rules(wm, wm_rules).drop("_dup_rank"), "warehouse_id")

# Route master
rm = (with_dup_rank(routes, ["route_id"], ["_ingestion_timestamp"])
        .join(F.broadcast(warehouse_ids), "origin_warehouse_id", "left"))
rm_rules = (id_format({"route_id": r"^ROUTE\d+$", "origin_warehouse_id": r"^ORIGIN_\d+$"}) + cast_failed()
          + not_null(["route_id", "origin_warehouse_id", "destination_city", "distance_km", "planned_hours"]) + flag_rule()
          + [rule("INVALID_DISTANCE", "RANGE", "ERROR", (F.col("distance_km") <= 0) | (F.col("distance_km") > MAX_DISTANCE_KM)),
             rule("INVALID_PLANNED_HOURS", "RANGE", "ERROR", (F.col("planned_hours") <= 0) | (F.col("planned_hours") > MAX_PLANNED_HRS)),
             rule("UNKNOWN_WAREHOUSE", "REFERENTIAL", "WARNING", F.col("_warehouse_found").isNull())]
          + duplicate(["_ingestion_timestamp"]))
validated["route_master"] = (apply_rules(rm, rm_rules).drop("_warehouse_found", "_dup_rank"), "route_id")

for n in ["carrier_master", "warehouse_master", "route_master"]:
    print(n, validated[n][0].groupBy("_is_valid").count().collect())

In [0]:
# Step 8 - Quarantine: invalid records with rejection reasons
for name, (df, _) in validated.items():
    (df.filter(~F.col("_is_valid"))
       .withColumn("rejection_reasons",
                   F.transform(F.filter("_dq_issues", lambda x: x["severity"] == "ERROR"), lambda x: x["rule_id"]))
       .withColumn("_quarantined_ts", F.current_timestamp())
       .drop("_is_valid")
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{SILVER}.quarantine_{name}"))
    print(f"Saved {SILVER}.quarantine_{name}")

# Issue log: one row per record per failed rule (errors and warnings)
issue_log = None
for name, (df, key) in validated.items():
    part = (df.select(F.lit(name).alias("dataset"), F.col(key).alias("record_id"), F.explode("_dq_issues").alias("i"))
              .select("dataset", "record_id", "i.rule_id", "i.category", "i.severity"))
    issue_log = part if issue_log is None else issue_log.unionByName(part)

(issue_log.withColumn("_logged_ts", F.current_timestamp())
          .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
          .saveAsTable(f"{SILVER}.dq_issue_log"))

display(spark.table(f"{SILVER}.quarantine_shipment_orders")
             .select("shipment_id", "rejection_reasons", "shipment_status", "package_weight_kg", "_source_file")
             .limit(50))

In [0]:
# Step 9 - Final Silver tables: valid records only, warnings kept as a flag column
for name, (df, _) in validated.items():
    (df.filter(F.col("_is_valid"))
       .withColumn("_dq_warnings",
                   F.transform(F.filter("_dq_issues", lambda x: x["severity"] == "WARNING"), lambda x: x["rule_id"]))
       .drop("_is_valid", "_dq_issues")
       .withColumn("_silver_validated_ts", F.current_timestamp())
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{SILVER}.{name}"))
    print(f"Saved {SILVER}.{name}")

In [0]:
# Step 10 - Quality report: accepted, rejected and issue counts
rows = []
for name, (df, _) in validated.items():
    inp = df.count()
    acc = spark.table(f"{SILVER}.{name}").count()
    rej = spark.table(f"{SILVER}.quarantine_{name}").count()
    warn = spark.table(f"{SILVER}.{name}").filter(F.size("_dq_warnings") > 0).count()
    rows.append((name, inp, acc, rej, warn, acc + rej == inp))

quality_report = (spark.createDataFrame(rows, "dataset string, input_rows long, accepted_rows long, rejected_rows long, "
                                              "accepted_with_warnings long, reconciles boolean")
                  .withColumn("rejection_pct", F.round(F.col("rejected_rows") * 100 / F.col("input_rows"), 2))
                  .withColumn("_report_ts", F.current_timestamp()))
quality_report.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.dq_quality_report")
display(quality_report)

# Issue counts per rule
display(spark.table(f"{SILVER}.dq_issue_log")
             .groupBy("dataset", "severity", "category", "rule_id")
             .agg(F.count("*").alias("issue_count"), F.countDistinct("record_id").alias("records"))
             .orderBy("dataset", "severity", F.desc("issue_count")))